In [0]:
import pandas as pd
import requests
import base64

github_token = "github_token_here"
repo_owner = "chaitanya1110-creates"
repo_name = "f1-project"
github_folder = "data" 
branch = "main"

tables_to_sync = {
    "workspace.f1_presentation.top_drivers_all_time": "top_drivers_all_time.csv",
    "workspace.f1_presentation.final_predictions": "final_predictions.csv",
    "workspace.f1_presentation.driver_bad_luck_stats": "driver_bad_luck_stats.csv"
}

headers = {"Authorization": f"token {github_token}"}

def delete_and_upload(spark_table, csv_name):
    path = f"{github_folder}/{csv_name}"
    url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/contents/{path}"
    
  
    res = requests.get(url, headers=headers)
    sha = res.json().get("sha") if res.status_code == 200 else None

    if sha:
        del_payload = {"message": f"Deleting old {csv_name}", "sha": sha, "branch": branch}
        requests.delete(url, headers=headers, json=del_payload)
        print(f"Deleted old {csv_name}")

    print(f"Fetching fresh data for {csv_name}...")
    pdf = spark.table(spark_table).toPandas()
    csv_content = pdf.to_csv(index=False)
    encoded_content = base64.b64encode(csv_content.encode("utf-8")).decode("utf-8")

    put_payload = {
        "message": f"Uploading fresh {csv_name}",
        "content": encoded_content,
        "branch": branch
    }
    put_res = requests.put(url, headers=headers, json=put_payload)
    
    if put_res.status_code in [200, 201]:
        print(f"{csv_name} is now updated.")
    else:
        print(f"Failed to upload {csv_name}: {put_res.text}")

for table, filename in tables_to_sync.items():
    try:
        delete_and_upload(table, filename)
    except Exception as e:
        print(f"Error with {filename}: {e}")


🗑️ Deleted old top_drivers_all_time.csv
📦 Fetching fresh data for top_drivers_all_time.csv...
✅ top_drivers_all_time.csv is now updated.
🗑️ Deleted old final_predictions.csv
📦 Fetching fresh data for final_predictions.csv...
✅ final_predictions.csv is now updated.
🗑️ Deleted old driver_bad_luck_stats.csv
📦 Fetching fresh data for driver_bad_luck_stats.csv...
✅ driver_bad_luck_stats.csv is now updated.

🚀 Sync complete. README.txt was not touched.
